# 🚗 CodeAlpha Task 3: Car Price Prediction with Machine Learning
**Author:** Data Science Intern  
**Domain:** Data Science & Machine Learning  
**Organization:** CodeAlpha  
**Date:** October 2026  

---
## 📌 1. Project Overview & Problem Statement
Accurately determining the fair market resale value of a pre-owned car is a crucial task for both used car buyers and dealerships. Second-hand car prices depend heavily on multi-dimensional variables including vehicle age, showroom sticker price, odometer mileage, fuel efficiency, transmission type, and previous ownership history.

### Objectives:
1. Perform **Exploratory Data Analysis (EDA)** to understand market depreciation trends.
2. Engineer predictive features such as **`Car_Age`** and **`Brand`**.
3. Build robust, leak-free preprocessing pipelines utilizing Scikit-Learn `ColumnTransformer`.
4. Train and benchmark 4 regression models: **Linear Regression**, **Decision Tree**, **Random Forest**, and **Gradient Boosting**.
5. Evaluate using standard regression metrics (**MAE**, **MSE**, **RMSE**, **R²**).
6. Interpret feature importance and explain market dynamics.

## 📂 2. Dataset Information & Source
- **Dataset Name:** Car Price Prediction Dataset (Vehicle dataset from CarDekho)
- **Source:** Public Domain (CC0: Public Domain)
- **Target Variable:** `Selling_Price` (Current resale price in ₹ Lakhs)
- **Key Features:** `Car_Name`, `Year`, `Present_Price`, `Driven_kms`, `Fuel_Type`, `Selling_type`, `Transmission`, `Owner`

In [ ]:
# Import Core Libraries
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Styling setup
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (9, 5), 'font.size': 11})

print('Libraries successfully imported.')

## 📥 3. Data Ingestion & Initial Inspection

In [ ]:
# Locate and load dataset
csv_paths = ['../data/car_data.csv', 'data/car_data.csv', '../data/car data.csv', 'data/car data.csv']
dataset_path = None
for p in csv_paths:
    if os.path.exists(p):
        dataset_path = p
        break

df = pd.read_csv(dataset_path)
print(f'Loaded dataset from: {dataset_path}')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

In [ ]:
# Dataset Metadata and Null Value Verification
print('--- Dataset Info ---')
df.info()

print('\n--- Missing Values Count ---')
print(df.isnull().sum())

print(f'\nExact Duplicate Rows: {df.duplicated().sum()}')

## 🧹 4. Data Cleaning & Feature Engineering
1. **Deduplication:** Remove exact duplicate rows.
2. **Vehicle Age (`Car_Age`):** Derived as `2024 - Year` to capture depreciation over time.
3. **Manufacturer Brand (`Brand`):** Extracted from `Car_Name` (e.g., Maruti Suzuki, Hyundai, Honda, Toyota, etc.).

In [ ]:
# Remove exact duplicates
df_cleaned = df.drop_duplicates().reset_index(drop=True)

# Compute Vehicle Age
CURRENT_YEAR = 2024
df_cleaned['Car_Age'] = CURRENT_YEAR - df_cleaned['Year']

# Extract Brand
def extract_brand(name):
    name = str(name).strip().lower()
    maruti = ['ritz', 'sx4', 'ciaz', 'wagon r', 'swift', 'vitara brezza', 's cross', 'alto 800', 'ertiga', 'dzire', 'alto k10', 'ignis', '800', 'baleno', 'omni']
    hyundai = ['i20', 'grand i10', 'i10', 'eon', 'xcent', 'elantra', 'creta', 'verna', 'santa fe']
    honda = ['city', 'brio', 'amaze', 'jazz', 'cr-v', 'civic', 'accord', 'activa']
    toyota = ['corolla altis', 'fortuner', 'innova', 'etios liva', 'etios cross', 'etios g', 'corolla', 'etios gd', 'camry', 'land cruiser']
    for m in maruti: 
        if name.startswith(m): return 'Maruti Suzuki'
    for h in hyundai:
        if name.startswith(h): return 'Hyundai'
    for c in honda:
        if name.startswith(c) or 'honda' in name: return 'Honda'
    for t in toyota:
        if name.startswith(t): return 'Toyota'
    if 'royal enfield' in name: return 'Royal Enfield'
    if 'bajaj' in name: return 'Bajaj'
    if 'yamaha' in name: return 'Yamaha'
    if 'tvs' in name: return 'TVS'
    if 'hero' in name: return 'Hero'
    if 'ktm' in name: return 'KTM'
    return 'Other'

df_cleaned['Brand'] = df_cleaned['Car_Name'].apply(extract_brand)
df_cleaned[['Car_Name', 'Brand', 'Year', 'Car_Age', 'Selling_Price', 'Present_Price']].head()

## 📊 5. Exploratory Data Analysis (EDA)
Visualizing the distributions, categorical breakdowns, and correlation matrix.

In [ ]:
# 1. Selling Price Distribution
plt.figure(figsize=(9, 4))
sns.histplot(df_cleaned['Selling_Price'], kde=True, color='#2b5c8f', bins=25)
plt.title('Distribution of Car Selling Prices (₹ in Lakhs)', fontweight='bold')
plt.xlabel('Selling Price (₹ in Lakhs)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# 2. Year vs Selling Price
plt.figure(figsize=(9, 4))
sns.regplot(data=df_cleaned, x='Year', y='Selling_Price', color='#1f77b4', line_kws={'color': '#e74c3c'})
plt.title('Manufacturing Year vs Selling Price', fontweight='bold')
plt.xlabel('Manufacturing Year')
plt.ylabel('Selling Price (₹ Lakhs)')
plt.show()

In [ ]:
# 3. Present Price vs Selling Price
plt.figure(figsize=(9, 4))
sns.scatterplot(data=df_cleaned, x='Present_Price', y='Selling_Price', hue='Fuel_Type', style='Transmission', s=70)
plt.title('Showroom Present Price vs Selling Price', fontweight='bold')
plt.xlabel('Present Price (₹ Lakhs)')
plt.ylabel('Selling Price (₹ Lakhs)')
plt.show()

In [ ]:
# 4. Correlation Heatmap
plt.figure(figsize=(7, 5))
num_cols = df_cleaned.select_dtypes(include=['int64', 'float64']).columns
sns.heatmap(df_cleaned[num_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Correlation Matrix of Numerical Features', fontweight='bold')
plt.show()

## ⚙️ 6. Preprocessing Pipeline & Data Splitting
To strictly prevent data leakage:
- Train-Test Split (80% train, 20% test, `random_state=42`) is executed first.
- `StandardScaler` scales numeric columns based on training data statistics.
- `OneHotEncoder` encodes categorical features without leaking test categories.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

features = ['Present_Price', 'Driven_kms', 'Car_Age', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner', 'Brand']
target = 'Selling_Price'

X = df_cleaned[features]
y = df_cleaned[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'X_train shape: {X_train.shape}, X_test shape: {X_test.shape}')

numeric_cols = ['Present_Price', 'Driven_kms', 'Car_Age', 'Owner']
categorical_cols = ['Fuel_Type', 'Selling_type', 'Transmission', 'Brand']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

## 🤖 7. Multi-Model Training & Evaluation
We benchmark four distinct regression models:
1. **Linear Regression** (Parametric Baseline)
2. **Decision Tree Regressor** (Non-linear Tree Model)
3. **Random Forest Regressor** (Ensemble Bagging)
4. **Gradient Boosting Regressor** (Ensemble Boosting)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

candidate_models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=8, min_samples_split=5),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10, min_samples_split=3),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
}

results = {}
fitted_pipes = {}

for name, model in candidate_models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('reg', model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    
    results[name] = {'Test R2': r2, 'MAE (₹ Lakhs)': mae, 'RMSE (₹ Lakhs)': rmse, 'MSE': mse}
    fitted_pipes[name] = pipe

results_df = pd.DataFrame(results).T
results_df.sort_values(by='Test R2', ascending=False)

## 📈 8. Diagnostic Visualizations & Residual Analysis

In [ ]:
best_name = 'Gradient Boosting'
best_model = fitted_pipes[best_name]
y_pred = best_model.predict(X_test)
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_test, y_pred, color='#3498db', alpha=0.75, edgecolors='k')
min_p, max_p = min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())
axes[0].plot([min_p, max_p], [min_p, max_p], color='#e74c3c', linestyle='--')
axes[0].set_title(f'Actual vs Predicted Selling Price ({best_name})', fontweight='bold')
axes[0].set_xlabel('Actual Price (₹ Lakhs)')
axes[0].set_ylabel('Predicted Price (₹ Lakhs)')

# Residual Distribution
sns.histplot(residuals, kde=True, color='#2ecc71', ax=axes[1])
axes[1].axvline(0, color='#e74c3c', linestyle='--')
axes[1].set_title('Residuals Distribution', fontweight='bold')
axes[1].set_xlabel('Residual Error (₹ Lakhs)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 🔍 9. Feature Importance Interpretation

In [ ]:
# Extract transformed feature names
cat_enc = best_model.named_steps['prep'].named_transformers_['cat']
encoded_cats = list(cat_enc.get_feature_names_out(categorical_cols))
all_features = numeric_cols + encoded_cats
importances = best_model.named_steps['reg'].feature_importances_

feat_df = pd.DataFrame({'Feature': all_features, 'Importance': importances})
feat_df = feat_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)

plt.figure(figsize=(9, 5))
sns.barplot(data=feat_df.head(8), x='Importance', y='Feature', palette='Blues_r')
plt.title(f'Top Influential Features ({best_name})', fontweight='bold')
plt.xlabel('Relative Importance')
plt.show()

feat_df.head(8)

## 💡 10. Key Insights & Conclusion

### Findings:
1. **`Present_Price` (Showroom Price) is the dominant predictor (~90% importance):** Cars with higher original purchase prices retain higher absolute valuations in the pre-owned market.
2. **Vehicle Age (`Car_Age`) is the second most critical factor (~7% importance):** Vehicles depreciate progressively year-over-year, following non-linear exponential decay.
3. **Gradient Boosting achieved the highest R² score (0.8288) and lowest MAE (₹1.008 Lakhs):** Outperforming standard linear regression and single decision trees.

### Limitations & Next Steps:
- Regional pricing variations (e.g. state road taxes) and vehicle physical condition/accident history are not present in this dataset.
- Expanding the dataset with newer vehicle models (2019–2026) will further improve model relevance for recent car models.